# Myeloid skin subset integration - Round 3

## Python modules

In [ ]:
import scanpy as sc
import anndata as ad

import scanorama
import scvi

from scipy import sparse
import pandas as pd
import numpy as np

import pickle

import seaborn as sb
import matplotlib.pyplot as plt

import os

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

In [ ]:
%load_ext rpy2.ipython

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Warnings R
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR) 

## R modules

In [ ]:
%%R
library(scran)

## Plotting 

In [ ]:
# Scanpy figures
sc.set_figure_params(figsize=(7, 7), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor="white")

In [ ]:
# Matplot figures
plt.rcParams['figure.figsize']=(7, 7)

In [ ]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

In [ ]:
# Set color function 
def set_color(adata, categories): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]

    for category in categories: 

        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)
        
    return(adata)

# Import AnnData 

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_2/myeloid/scvi.h5ad')
obs_names = adata.obs_names
var_names = adata.var_names

In [ ]:
sc.pl.umap(adata, color=['leiden', 'msS_RNA', 'msG2M_RNA', 'msCC_diff_RNA', 'pMT_RNA', 'pRP_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)

In [ ]:
adata = sc.read_h5ad('data/object/pp.h5ad')
adata = adata[obs_names, var_names].copy()

In [ ]:
adata = adata[adata.obs['hto_demux_class']=='Skin']
adata = adata[:, (adata.X>=3).sum(axis=0)>=3]

# Processing functions 

## SCVI integration 

In [ ]:
def scvi_integrate(adata, hvg, cat=None, con=None, model_dir=None, file=None): 
    
    # Subset by hvg 
    adata = adata[:, adata.var_names.isin(hvg)].copy()
    
    scvi.model.SCVI.setup_anndata(
        adata,
        batch_key='sample_name', 
        categorical_covariate_keys=cat, 
        continuous_covariate_keys=con
    )
    
    model = scvi.model.SCVI(
        adata, 
        n_latent=10, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'
    )
    
    model.train()
    
    # Write output 
    if model_dir is not None: 
        model.save(model_dir, overwrite=True)
    
    adata.obsm['X_scvi'] = model.get_latent_representation()
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return adata

## Dim reduction and cluster function 

In [ ]:
def dim_clust(adata, use_rep=None, n_neighbors=15, resolution=1, min_dist=0.5, metric='euclidean', file=None): 
    
    # Dim reduction and clustering on the scanorama corrected embedding 
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, metric=metric)
    sc.tl.leiden(adata, resolution=resolution)
    
    sc.tl.umap(adata, min_dist=min_dist)
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return(adata)

# SCVI with continuous covariates

In [ ]:
adata_0 = scvi_integrate(adata.copy(), hvg=adata.var_names, cat=None, con=['msS_RNA', 'msG2M_RNA', 'pMT_RNA'], model_dir='data/object/subsets/round_3/myeloid_skin/model_con', file='data/object/subsets/round_3/myeloid_skin/scvi_con.h5ad')
adata_0 = dim_clust(adata_0, use_rep='X_scvi', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_3/myeloid_skin/scvi_con.h5ad')

In [ ]:
sc.pl.umap(adata_0, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)

# SCVI with categorical and continuous covariates

In [ ]:
adata_1 = scvi_integrate(adata.copy(), hvg=adata.var_names, cat=['patient_id', 'genotype_class', 'time_point'], con=['msS_RNA', 'msG2M_RNA', 'pMT_RNA'], model_dir='data/object/subsets/round_3/myeloid_skin/model_con_cat', file='data/object/subsets/round_3/myeloid_skin/scvi_con_cat.h5ad')
adata_1 = dim_clust(adata_1, use_rep='X_scvi', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_3/myeloid_skin/scvi_con_cat.h5ad')

In [ ]:
sc.pl.umap(adata_1, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)